# Análise de Infraestrutura Computacional
**Projeto:** Análise da Capacidade Computacional e Distribuição de Processadores  
**Disciplina:** Big Data - Python   
**Base de dados:** Inventario.xlsx — 991 registros  

---
## Etapas
1. Importação e leitura da base
2. Tratamento e criação de colunas derivadas
3. Criação do dado simulado (Impacto_Operacional)
4. Exportação da base tratada

---
## 1. Importação das bibliotecas

In [1]:
import pandas as pd
import re

---
## 2. Leitura da base original

In [2]:
df = pd.read_excel('../data/raw/Inventario.xlsx')

print(f'Shape: {df.shape}')
print(f'Colunas: {list(df.columns)}')
df.head()

Shape: (991, 6)
Colunas: ['Last inventory', 'Operating system', 'RAM (MB)', 'CPU (MHz)', 'CPU type', 'CPU number']


,Last inventory,Operating system,RAM (MB),CPU (MHz),CPU type,CPU number
0,2025-02-06 15:37:48,Microsoft Windows Server 2022 Standard,8192,2793,Intel(R) Xeon(R) Silver 4309Y CPU @ 2.80GHz [2...,1
1,2025-02-26 15:01:49,Microsoft Windows Server 2019 Standard,8192,2394,Intel(R) Xeon(R) Platinum 8260 CPU @ 2.40GHz [...,4
2,2025-07-09 09:15:12,Microsoft Windows Server 2022 Standard,16384,2793,Intel(R) Xeon(R) Silver 4309Y CPU @ 2.80GHz [2...,1
3,2025-10-24 10:13:29,Microsoft Windows Server 2022 Datacenter,131072,2600,INTEL(R) XEON(R) SILVER 4509Y [8 core(s) x86_64],2
4,2025-11-13 13:18:16,Microsoft Windows Server 2019 Standard Evaluation,16384,3301,Intel(R) Core(TM) i5-4590 CPU @ 3.30GHz [4 cor...,1


In [3]:
# Verificar valores nulos
print('Valores nulos por coluna:')
print(df.isnull().sum())

Valores nulos por coluna:
Last inventory      0
Operating system    0
RAM (MB)            0
CPU (MHz)           0
CPU type            0
CPU number          0
dtype: int64


---
## 3. Tratamento dos dados
### 3.1 Converter RAM de MB para GB

In [4]:
df['RAM_GB'] = df['RAM (MB)'] / 1024

print('Distribuição de RAM (GB):')
print(df['RAM_GB'].value_counts().sort_index())

Distribuição de RAM (GB):
RAM_GB
5.714844        1
6.823242      161
7.683594        1
7.751953        1
7.999023        1
8.000000      223
10.000000       1
12.000000       3
16.000000     516
16.062500       1
20.000000       1
24.000000       9
32.000000      37
48.000000       1
64.000000      28
128.000000      6
Name: count, dtype: int64


### 3.2 Extrair família de CPU

In [5]:
def get_cpu_family(cpu_str):
    """
    Extrai a família do processador a partir da string bruta do campo CPU type.
    Retorna categorias como: Xeon Silver, Core i5, Ryzen 5, etc.
    """
    cpu = cpu_str.upper()
    if 'XEON' in cpu:
        if 'PLATINUM' in cpu: return 'Xeon Platinum'
        if 'GOLD' in cpu:     return 'Xeon Gold'
        if 'SILVER' in cpu:   return 'Xeon Silver'
        return 'Xeon Outro'
    if 'RYZEN 9' in cpu: return 'Ryzen 9'
    if 'RYZEN 7' in cpu: return 'Ryzen 7'
    if 'RYZEN 5' in cpu: return 'Ryzen 5'
    if 'RYZEN 3' in cpu: return 'Ryzen 3'
    if 'I9' in cpu: return 'Core i9'
    if 'I7' in cpu: return 'Core i7'
    if 'I5' in cpu: return 'Core i5'
    if 'I3' in cpu: return 'Core i3'
    return 'Outro'

df['CPU_Family'] = df['CPU type'].apply(get_cpu_family)

print('Distribuição por família de CPU:')
print(df['CPU_Family'].value_counts())

Distribuição por família de CPU:
CPU_Family
Core i5          401
Ryzen 5          327
Core i7          102
Ryzen 7           73
Xeon Platinum     26
Core i9           18
Xeon Outro        14
Xeon Silver       13
Xeon Gold         11
Ryzen 3            6
Name: count, dtype: int64


### 3.3 Extrair número de núcleos por CPU e calcular total

In [6]:
def get_cores_per_cpu(cpu_str):
    """
    Extrai a quantidade de núcleos por CPU a partir do padrão [X core(s)] 
    presente na string da coluna CPU type.
    """
    match = re.search(r'\[(\d+)\s+core', cpu_str, re.IGNORECASE)
    return int(match.group(1)) if match else 1

df['Cores_per_CPU'] = df['CPU type'].apply(get_cores_per_cpu)

# Total de núcleos = núcleos por CPU × quantidade de CPUs físicas
df['Total_Cores'] = df['Cores_per_CPU'] * df['CPU number']

print('Estatísticas de núcleos totais:')
print(df['Total_Cores'].describe())

Estatísticas de núcleos totais:
count    991.000000
mean       7.082745
std        4.161295
min        1.000000
25%        4.000000
50%        6.000000
75%        8.000000
max       64.000000
Name: Total_Cores, dtype: float64


### 3.4 Classificar perfil da máquina (Machine_Profile)

In [7]:
# Sufixos de CPU móvel usados para identificar notebooks
MOBILE_SUFFIXES = [
    '1255U', '1235U', '1035G1', '1135G7',
    '5500U', '5875U', '7735HS', '12700H', '11800H'
]

def get_machine_profile(row):
    """
    Classifica a máquina como Servidor, Notebook ou Desktop
    com base no sistema operacional e no modelo de CPU.
    Nota: classificação estimada — a base não possui campo explícito de tipo de equipamento.
    """
    os_str = str(row['Operating system']).upper()
    cpu_str = str(row['CPU type']).upper()

    if 'SERVER' in os_str or 'UBUNTU' in os_str:
        return 'Servidor'

    if any(suffix in cpu_str for suffix in MOBILE_SUFFIXES):
        return 'Notebook'

    if 'WINDOWS 11' in os_str or 'WINDOWS 10' in os_str:
        return 'Desktop'

    return 'Indefinido'

df['Machine_Profile'] = df.apply(get_machine_profile, axis=1)

print('Distribuição por perfil da máquina:')
print(df['Machine_Profile'].value_counts())

Distribuição por perfil da máquina:
Machine_Profile
Notebook    812
Desktop     112
Servidor     67
Name: count, dtype: int64


---
## 4. Dado simulado — Impacto_Operacional

Como a base não possui métricas de uso real, foi criado um indicador estimado de impacto operacional  
com base nas especificações de hardware disponíveis (RAM, núcleos e família de CPU).

**Critério de pontuação:**
| Critério | Pontos |
|---|---|
| RAM ≥ 32 GB | +3 |
| RAM entre 16 e 31 GB | +2 |
| RAM < 16 GB | +1 |
| Total de núcleos ≥ 8 | +3 |
| Total de núcleos entre 4 e 7 | +2 |
| Total de núcleos < 4 | +1 |
| CPU Xeon Platinum / Gold | +3 |
| CPU Xeon Silver / Core i7 / Core i9 / Ryzen 7 / Ryzen 9 | +2 |
| Demais CPUs | +1 |

**Classificação final:**
- Score ≥ 7 → **Baixo** impacto operacional (máquina robusta)
- Score entre 5 e 6 → **Médio** impacto operacional
- Score ≤ 4 → **Alto** impacto operacional (máquina mais limitada)

In [8]:
ALTA_PERFORMANCE = ['Xeon Platinum', 'Xeon Gold']
MEDIA_PERFORMANCE = ['Xeon Silver', 'Core i7', 'Core i9', 'Ryzen 7', 'Ryzen 9']

def get_impacto_operacional(row):
    """
    Calcula o impacto operacional estimado da máquina com base em
    RAM, total de núcleos e família de CPU.
    Retorna: Alto, Médio ou Baixo.
    """
    score = 0

    # RAM
    if row['RAM_GB'] >= 32:   score += 3
    elif row['RAM_GB'] >= 16: score += 2
    else:                     score += 1

    # Núcleos
    if row['Total_Cores'] >= 8:   score += 3
    elif row['Total_Cores'] >= 4: score += 2
    else:                         score += 1

    # CPU
    if row['CPU_Family'] in ALTA_PERFORMANCE:   score += 3
    elif row['CPU_Family'] in MEDIA_PERFORMANCE: score += 2
    else:                                        score += 1

    # Score alto = máquina mais robusta = menor impacto operacional
    if score >= 7:   return 'Baixo'
    elif score >= 5: return 'Médio'
    else:            return 'Alto'

df['Impacto_Operacional'] = df.apply(get_impacto_operacional, axis=1)

print('Distribuição por Impacto Operacional:')
print(df['Impacto_Operacional'].value_counts())

Distribuição por Impacto Operacional:
Impacto_Operacional
Médio    406
Alto     370
Baixo    215
Name: count, dtype: int64


---
## 5. Visão geral da base tratada

In [9]:
print(f'Total de registros: {len(df)}')
print(f'Colunas finais: {list(df.columns)}')
df.head(10)

Total de registros: 991
Colunas finais: ['Last inventory', 'Operating system', 'RAM (MB)', 'CPU (MHz)', 'CPU type', 'CPU number', 'RAM_GB', 'CPU_Family', 'Cores_per_CPU', 'Total_Cores', 'Machine_Profile', 'Impacto_Operacional']


,Last inventory,Operating system,RAM (MB),CPU (MHz),CPU type,CPU number,RAM_GB,CPU_Family,Cores_per_CPU,Total_Cores,Machine_Profile,Impacto_Operacional
0,2025-02-06 15:37:48,Microsoft Windows Server 2022 Standard,8192,2793,Intel(R) Xeon(R) Silver 4309Y CPU @ 2.80GHz [2...,1,8.0,Xeon Silver,2,2,Servidor,Alto
1,2025-02-26 15:01:49,Microsoft Windows Server 2019 Standard,8192,2394,Intel(R) Xeon(R) Platinum 8260 CPU @ 2.40GHz [...,4,8.0,Xeon Platinum,1,4,Servidor,Médio
2,2025-07-09 09:15:12,Microsoft Windows Server 2022 Standard,16384,2793,Intel(R) Xeon(R) Silver 4309Y CPU @ 2.80GHz [2...,1,16.0,Xeon Silver,2,2,Servidor,Médio
3,2025-10-24 10:13:29,Microsoft Windows Server 2022 Datacenter,131072,2600,INTEL(R) XEON(R) SILVER 4509Y [8 core(s) x86_64],2,128.0,Xeon Silver,8,16,Servidor,Baixo
4,2025-11-13 13:18:16,Microsoft Windows Server 2019 Standard Evaluation,16384,3301,Intel(R) Core(TM) i5-4590 CPU @ 3.30GHz [4 cor...,1,16.0,Core i5,4,4,Servidor,Médio
5,2025-11-13 15:36:52,Microsoft Windows Server 2019 Standard Evaluation,16384,3301,Intel(R) Core(TM) i5-4590 CPU @ 3.30GHz [4 cor...,1,16.0,Core i5,4,4,Servidor,Médio
6,2025-12-02 10:38:05,Microsoft Windows Server 2019 Standard,131072,2000,INTEL(R) XEON(R) SILVER 4514Y [16 core(s) x86_64],2,128.0,Xeon Silver,16,32,Servidor,Baixo
7,2025-12-02 10:42:01,Microsoft Windows Server 2019 Standard,16384,2000,INTEL(R) XEON(R) SILVER 4514Y [6 core(s) x86_64],1,16.0,Xeon Silver,6,6,Servidor,Médio
8,2025-12-04 14:27:41,Microsoft Windows Server 2022 Standard,131072,2793,Intel(R) Xeon(R) Silver 4309Y CPU @ 2.80GHz [8...,1,128.0,Xeon Silver,8,8,Servidor,Baixo
9,2025-12-16 15:03:08,Microsoft Windows Server 2022 Datacenter,131072,2600,INTEL(R) XEON(R) SILVER 4509Y [8 core(s) x86_64],2,128.0,Xeon Silver,8,16,Servidor,Baixo


In [10]:
# KPI 1 — CPU mais utilizada
kpi_cpu = df['CPU_Family'].value_counts().idxmax()
kpi_cpu_pct = df['CPU_Family'].value_counts(normalize=True).max() * 100
print(f'KPI 1 — CPU mais utilizada: {kpi_cpu} ({kpi_cpu_pct:.1f}% da infraestrutura)')

# KPI 2 — % de máquinas com alto impacto operacional
kpi_alto = (df['Impacto_Operacional'] == 'Alto').mean() * 100
print(f'KPI 2 — Máquinas com alto impacto operacional: {kpi_alto:.1f}%')

KPI 1 — CPU mais utilizada: Core i5 (40.5% da infraestrutura)
KPI 2 — Máquinas com alto impacto operacional: 37.3%


---
## 6. Exportar base tratada

In [11]:
df.to_csv('../data/processed/inventario_tratado.csv', index=False, encoding='utf-8-sig')
print('Base tratada exportada com sucesso: data/processed/inventario_tratado.csv')
print(f'Shape final: {df.shape}')

Base tratada exportada com sucesso: data/processed/inventario_tratado.csv
Shape final: (991, 12)
